#Distorted Visual Sequence Pattern Recognition using Deep Learning


**Architecture**: ResNet-style CNN Backbone → BiLSTM → CTC Loss  


In [2]:
import os, glob, random, string, time
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image, ImageFilter, ImageEnhance
import editdistance

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import cv2
import torchvision.transforms as T

In [17]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.11.0+cu128
CUDA available: True


In [5]:
import zipfile
import os

zip_path = "cig_ps.zip"
if os.path.exists(zip_path):
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(".")
        print("Extraction successful!")
    except zipfile.BadZipFile:
        print("Error: The file is not a valid zip file. Please re-upload a correct zip archive.")
    except Exception as e:
        print(f"An unexpected error occurred during extraction: {e}")
else:
    print("Zip file not found in directory. Please re-upload.")

Extraction successful!


**CONFIG**

In [18]:
TRAIN_IMG_DIR   = "/content/cig_ps/train_images"
TRAIN_LABEL_CSV = "/content/cig_ps/train-labels.csv"
TEST_IMG_DIR    = "/content/cig_ps/test_images"
YOUR_NAME       = "Tanvi_Mandan"
YOUR_ENROLL     = "24118072"

IMG_H, IMG_W = 64, 256

CHARS       = string.ascii_uppercase + string.digits
BLANK_IDX   = 0
CHAR2IDX    = {c: i+1 for i, c in enumerate(CHARS)}
IDX2CHAR    = {i+1: c for i, c in enumerate(CHARS)}
NUM_CLASSES = len(CHARS) + 1

BATCH_SIZE   = 128
NUM_EPOCHS   = 80
LR           = 3e-4
WEIGHT_DECAY = 1e-4
VAL_SPLIT    = 0.1
PATIENCE     = 15

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


**LABEL ENCODING DECODING**

In [7]:
def encode_label(text):
    return [CHAR2IDX[c] for c in text.upper() if c in CHAR2IDX]

def decode_ctc_greedy(indices):
    out, prev = [], None
    for idx in indices:
        if idx != prev:
            if idx != BLANK_IDX:
                out.append(IDX2CHAR.get(idx, "?"))
            prev = idx
    return "".join(out)

def greedy_decode_batch(log_probs):
    preds = log_probs.argmax(2).permute(1, 0)   # (B, T)
    return [decode_ctc_greedy(p.tolist()) for p in preds]

def compute_cer(preds, targets):
    total_dist = sum(editdistance.eval(p, t) for p, t in zip(preds, targets))
    total_len  = max(sum(len(t) for t in targets), 1)
    return total_dist / total_len

**DATSET AND AUGUMENTATION**

In [8]:
import torchvision.transforms as T

train_transforms = T.Compose([
    T.RandomAffine(degrees=5, translate=(0.05, 0.05), scale=(0.9, 1.1)),

    T.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5)),

    T.ColorJitter(brightness=0.2, contrast=0.2),

    T.ToTensor(),

    T.RandomErasing(p=0.2, scale=(0.02, 0.1), ratio=(0.3, 3.3))
])

In [9]:
class CaptchaDataset(Dataset):
    def __init__(self, img_dir, labels=None, augment=False):
        self.img_dir = img_dir
        self.labels  = labels
        self.augment = augment
        if labels is None:
            self.files = sorted(
                glob.glob(os.path.join(img_dir, "*.png")) +
                glob.glob(os.path.join(img_dir, "*.jpg"))
            )
        else:
            self.files = [os.path.join(img_dir, f) for f, _ in labels]


        self.base_tf = T.Compose([
            T.Grayscale(1),
            T.Resize((IMG_H, IMG_W)),
            T.ToTensor(),
            T.Normalize([0.5], [0.5]),


            ])

    def __len__(self):
        return len(self.files)

    def _augment_pil(self, img: Image.Image) -> Image.Image:
        if random.random() < 0.4:
            angle = random.uniform(-5, 5)
            img = img.rotate(angle, fillcolor=255)
        if random.random() < 0.4:
            img = ImageEnhance.Brightness(img).enhance(random.uniform(0.7, 1.3))
        if random.random() < 0.4:
            img = ImageEnhance.Contrast(img).enhance(random.uniform(0.7, 1.3))
        return img

    def _augment_np(self, img_np: np.ndarray) -> np.ndarray:

        # Gaussian blur
        if random.random() < 0.3:
            k = random.choice([3, 5])
            img_np = cv2.GaussianBlur(img_np, (k, k), 0)

        # Additive noise
        if random.random() < 0.4:
            noise = np.random.randint(-20, 20, img_np.shape, dtype=np.int16)
            img_np = np.clip(img_np.astype(np.int16) + noise, 0, 255).astype(np.uint8)

        # Morphological ops
        if random.random() < 0.25:
            kernel = np.ones((2, 2), np.uint8)
            op = random.choice([cv2.erode, cv2.dilate])
            img_np = op(img_np, kernel, iterations=1)

        # Horizontal shift
        if random.random() < 0.3:
            shift = int(IMG_W * 0.05 * random.uniform(-1, 1))
            M = np.float32([[1, 0, shift], [0, 1, 0]])
            img_np = cv2.warpAffine(img_np, M, (IMG_W, IMG_H),
                                    borderMode=cv2.BORDER_REPLICATE)
        # Random cutout column
        if random.random() < 0.2:
            col_start = random.randint(0, IMG_W - 20)
            col_w     = random.randint(5, 20)
            img_np[:, col_start:col_start+col_w] = 200
        return img_np

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("L")

        if self.augment:
            img = self._augment_pil(img)
            img_np = self._augment_np(
                np.array((img)))

            img = Image.fromarray(img_np)

        img_t = self.base_tf(img)

        if self.labels is None:
            return img_t, os.path.basename(self.files[idx])

        _, label_str = self.labels[idx]
        label_str = label_str.upper()
        return img_t, torch.tensor(encode_label(label_str), dtype=torch.long), label_str


def collate_train(batch):
    imgs, labels, strs = zip(*batch)
    return (
        torch.stack(imgs),
        torch.cat(labels),
        torch.tensor([len(l) for l in labels], dtype=torch.long),
        list(strs),
    )

def collate_test(batch):
    imgs, fnames = zip(*batch)
    return torch.stack(imgs), list(fnames)

#**MODEL- RSNET & CRNN**

In [10]:
import torch.nn as nn

class ConvBNReLU(nn.Module):
    def __init__(self, in_c, out_c, k=3, s=1, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, k, s, p, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class ResBlock(nn.Module):
    def __init__(self, channels, dropout=0.1):
        super().__init__()
        self.conv1 = ConvBNReLU(channels, channels)
        self.conv2 = nn.Sequential(
            nn.Conv2d(channels, channels, 3, 1, 1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.drop  = nn.Dropout2d(dropout)
        self.relu  = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(x + self.drop(self.conv2(self.conv1(x))))


class SEBlock(nn.Module):
    def __init__(self, channels, r=16):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, channels // r, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // r, channels, bias=False),
            nn.Sigmoid(),
        )
    def forward(self, x):
        w = self.se(x).view(x.size(0), x.size(1), 1, 1)
        return x * w


class ResNetCRNN(nn.Module):
    def __init__(self, num_classes, rnn_hidden=128, rnn_layers=1, dropout=0.2):  # Reduced hidden size & layers
        super().__init__()
        # Stem: Down from 64 to 32 channels
        self.stem = nn.Sequential(
            ConvBNReLU(1, 16, k=3, s=1, p=1),
            ConvBNReLU(16, 32, k=3, s=1, p=1),
            nn.MaxPool2d(2, 2),
        )
        # Stage 1
        self.stage1 = nn.Sequential(
            ConvBNReLU(32, 64),
            ResBlock(64, dropout=0.05),
            SEBlock(64),
            nn.MaxPool2d(2, 2),
        )
        # Stage 2
        self.stage2 = nn.Sequential(
            ConvBNReLU(64, 128),
            ResBlock(128, dropout=0.1),
            SEBlock(128),
            nn.MaxPool2d(2, 2),
        )
        # Stage 3
        self.stage3 = nn.Sequential(
            ConvBNReLU(128, 256),  # Capped at 256 instead of 512
            ResBlock(256, dropout=0.1),
            SEBlock(256),
            nn.MaxPool2d((2, 1), (2, 1)),
        )
        # Stage 4
        self.stage4 = nn.Sequential(
            ConvBNReLU(256, 256),
            ResBlock(256, dropout=0.1),
            SEBlock(256),
            nn.MaxPool2d((2, 1), (2, 1)),
            ConvBNReLU(256, 256),
            nn.AdaptiveAvgPool2d((1, None)),
        )
        # BiLSTM
        self.rnn = nn.LSTM(
            256, rnn_hidden, rnn_layers,
            batch_first=False, bidirectional=True,
            dropout=dropout if rnn_layers > 1 else 0.0,
        )
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(rnn_hidden * 2, num_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LSTM):
                for name, p in m.named_parameters():
                    if "weight_ih" in name: nn.init.xavier_uniform_(p.data)
                    elif "weight_hh" in name: nn.init.orthogonal_(p.data)
                    elif "bias" in name: nn.init.zeros_(p.data)

    def forward(self, x):
        f = self.stem(x)
        f = self.stage1(f)
        f = self.stage2(f)
        f = self.stage3(f)
        f = self.stage4(f)
        f = f.squeeze(2)
        f = f.permute(2, 0, 1)
        out, _ = self.rnn(f)
        out = self.fc(self.drop(out))
        return F.log_softmax(out, dim=2)


# shape test
with torch.no_grad():
    _m = ResNetCRNN(NUM_CLASSES)
    _x = torch.randn(2, 1, IMG_H, IMG_W)
    _y = _m(_x)
    print(f"Model output shape: {_y.shape}  (T={_y.shape[0]}, B=2, C={_y.shape[2]})")
    total_params = sum(p.numel() for p in _m.parameters())
    print(f"Total parameters: {total_params:,}")
del _m, _x, _y

Model output shape: torch.Size([32, 2, 37])  (T=32, B=2, C=37)
Total parameters: 4,727,957


**Loss & training utilities**

In [28]:
ctc_loss_fn = nn.CTCLoss(blank=BLANK_IDX, reduction="mean", zero_infinity=True)


# Initialize the GradScaler for AMP global scope
scaler = torch.amp.GradScaler('cuda')

def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = total_cer = n = 0
    for imgs, lbl, llen, strs in loader:
        imgs = imgs.to(DEVICE)
        lbl  = lbl.to(DEVICE)
        llen = llen.to(DEVICE)

        optimizer.zero_grad()

        # Casts operations to mixed precision
        with torch.amp.autocast('cuda'):
            log_probs = model(imgs)
            B = imgs.size(0)
            T = log_probs.size(0)
            input_len = torch.full((B,), T, dtype=torch.long, device=DEVICE)
            loss = ctc_loss_fn(log_probs, lbl, input_len, llen)

        # Scales the loss and calls backward
        scaler.scale(loss).backward()

        # Unscales gradients before clipping
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)

        # Optimizer step
        scaler.step(optimizer)
        scaler.update()

        with torch.no_grad():
            preds = greedy_decode_batch(log_probs.detach().cpu())
            total_loss += loss.item() * B
            total_cer  += compute_cer(preds, strs) * B
            n += B

    return total_loss / n, total_cer / n


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = total_cer = n = 0
    for imgs, lbl, llen, strs in loader:
        imgs = imgs.to(DEVICE)
        lbl  = lbl.to(DEVICE)
        llen = llen.to(DEVICE)

        log_probs = model(imgs)
        B  = imgs.size(0)
        T  = log_probs.size(0)
        input_len = torch.full((B,), T, dtype=torch.long, device=DEVICE)

        total_loss += ctc_loss_fn(log_probs, lbl, input_len, llen).item() * B
        total_cer  += compute_cer(greedy_decode_batch(log_probs.cpu()), strs) * B
        n += B

    return total_loss / n, total_cer / n

In [23]:
def validate(model, dataloader, device):
    model.eval()
    total_cer = 0
    total_samples = 0

    with torch.no_grad():
        for images, targets in dataloader:
            images = images.to(device)

            output = model(images)
            _, preds = output.max(2)
            preds = preds.transpose(0, 1)

            for i in range(len(preds)):

                pred_text = "".join([chr(c + 65) for c in preds[i] if c != 0])
                true_text = targets[i]

                dist = editdistance.eval(pred_text, true_text)
                total_cer += dist / max(len(true_text), 1)
                total_samples += 1

    return total_cer / total_samples

**Test Time Augmentation**

In [24]:
@torch.no_grad()
def predict_with_tta(model, imgs_cpu, n_aug=4):
    model.eval()
    B = imgs_cpu.size(0)
    sum_probs = None
    lp = model(imgs_cpu.to(DEVICE)).cpu()
    sum_probs = torch.exp(lp)

    aug_tf = T.Compose([
        T.RandomApply([T.GaussianBlur(3, sigma=(0.1, 1.0))], p=0.5),
        T.RandomAffine(degrees=3, translate=(0.03, 0.0), fill=-1.0),
    ])
    for _ in range(n_aug - 1):
        aug_imgs = torch.stack([aug_tf(imgs_cpu[i]) for i in range(B)])
        lp_aug   = model(aug_imgs.to(DEVICE)).cpu()
        sum_probs = sum_probs + torch.exp(lp_aug)

    avg_log_probs = torch.log(sum_probs / n_aug + 1e-9)
    return greedy_decode_batch(avg_log_probs)

**Main training loop**

In [25]:
def main(predict_only=False):
    model = ResNetCRNN(NUM_CLASSES).to(DEVICE)

    if predict_only:
        model.load_state_dict(torch.load("best_crnn_model.pth", map_location=DEVICE))
        print("Loaded best_crnn_model.pth for prediction.")
    else:
        df = pd.read_csv(TRAIN_LABEL_CSV)
        all_labels = list(zip(df["image"].astype(str), df["text"].astype(str)))
        random.shuffle(all_labels)

        val_n     = int(len(all_labels) * VAL_SPLIT)
        train_data = all_labels[val_n:]
        val_data   = all_labels[:val_n]

        kw = dict(num_workers=2, pin_memory=True, persistent_workers=True)
        train_loader = DataLoader(
            CaptchaDataset(TRAIN_IMG_DIR, train_data, augment=True),
            BATCH_SIZE, shuffle=True, collate_fn=collate_train, **kw
        )
        val_loader = DataLoader(
            CaptchaDataset(TRAIN_IMG_DIR, val_data, augment=False),
            BATCH_SIZE, shuffle=False, collate_fn=collate_train, **kw
        )

        print(f"Train: {len(train_data)}  |  Val: {len(val_data)}")

        #  Optimiser & scheduler
        optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

        scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=20, T_mult=2, eta_min=1e-6
        )

        # Training
        best_cer, patience_cnt = float("inf"), 0
        print(f"{'Ep':>4}  {'TrLoss':>8}  {'TrCER':>7}  {'VlLoss':>8}  {'VlCER':>7}  {'LR':>9}")
        print("-" * 55)

        for ep in range(1, NUM_EPOCHS + 1):
            t0 = time.time()
            tr_loss, tr_cer = train_one_epoch(model, train_loader, optimizer)
            vl_loss, vl_cer = evaluate(model, val_loader)
            scheduler.step()

            cur_lr = optimizer.param_groups[0]["lr"]
            elapsed = time.time() - t0
            print(f"{ep:>4}  {tr_loss:>8.4f}  {tr_cer:>7.4f}  "
                  f"{vl_loss:>8.4f}  {vl_cer:>7.4f}  {cur_lr:>9.2e}  [{elapsed:.0f}s]")

            if vl_cer < best_cer:
                best_cer = vl_cer
                patience_cnt = 0
                torch.save(model.state_dict(), "best_crnn_model.pth")
                print(f"Best model saved  (Val CER={best_cer:.4f})")
            else:
                patience_cnt += 1
                if patience_cnt >= PATIENCE:
                    print(f"Early stop at epoch {ep}  (no improvement for {PATIENCE} epochs).")
                    break

        print(f"\nBest Val CER: {best_cer:.4f}")
        model.load_state_dict(torch.load("best_crnn_model.pth", map_location=DEVICE))

    # Inference
    test_ds     = CaptchaDataset(TEST_IMG_DIR, labels=None, augment=False)
    test_loader = DataLoader(
        test_ds, BATCH_SIZE, shuffle=False,
        collate_fn=collate_test, num_workers=2
    )

    model.eval()
    results = []
    for imgs, fnames in test_loader:
        preds = predict_with_tta(model, imgs, n_aug=4)
        results.extend(zip(fnames, preds))

    out_csv = f"submission_{YOUR_NAME}_{YOUR_ENROLL}.csv"
    pd.DataFrame(results, columns=["image", "prediction"]).to_csv(out_csv, index=False)
    print(f"\nSubmission saved: {out_csv}  ({len(results)} rows)")
    return out_csv


out_csv = main(predict_only=False)

Train: 18000  |  Val: 2000
  Ep    TrLoss    TrCER    VlLoss    VlCER         LR
-------------------------------------------------------


/tmp/ipykernel_14429/1190970065.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_14429/1190970065.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_14429/1190970065.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


   1    4.2955   1.0490    3.7198   1.0000   2.98e-04  [33s]
Best model saved  (Val CER=1.0000)


/tmp/ipykernel_14429/1190970065.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


   2    3.6716   0.9844    3.5890   0.9319   2.93e-04  [33s]
Best model saved  (Val CER=0.9319)
   3    3.5340   0.9277    3.4989   0.9180   2.84e-04  [29s]
Best model saved  (Val CER=0.9180)
   4    3.3382   0.8948    3.1637   0.8628   2.71e-04  [28s]
Best model saved  (Val CER=0.8628)
   5    3.0297   0.8550    2.7192   0.7996   2.56e-04  [29s]
Best model saved  (Val CER=0.7996)
   6    2.6602   0.7953    2.3576   0.7365   2.38e-04  [28s]
Best model saved  (Val CER=0.7365)
   7    2.2982   0.7139    1.9954   0.6418   2.18e-04  [28s]
Best model saved  (Val CER=0.6418)
   8    1.9632   0.6170    1.6074   0.5010   1.97e-04  [28s]
Best model saved  (Val CER=0.5010)
   9    1.6627   0.5117    1.3992   0.4203   1.74e-04  [30s]
Best model saved  (Val CER=0.4203)
  10    1.3993   0.4076    1.1336   0.3094   1.50e-04  [28s]
Best model saved  (Val CER=0.3094)
  11    1.1917   0.3299    0.8473   0.1991   1.27e-04  [29s]
Best model saved  (Val CER=0.1991)
  12    1.0239   0.2653    0.6872   0.14

...

  40    0.1241   0.0290    0.0145   0.0009   1.50e-04  [28s]
Best model saved  (Val CER=0.0009)
  41    0.1196   0.0272    0.0138   0.0013   1.39e-04  [28s]


/tmp/ipykernel_14429/1190970065.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  42    0.1147   0.0271    0.0117   0.0009   1.27e-04  [29s]


/tmp/ipykernel_14429/1190970065.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  43    0.1131   0.0266    0.0127   0.0014   1.16e-04  [28s]
  44    0.1100   0.0264    0.0113   0.0014   1.04e-04  [28s]
  45    0.1110   0.0262    0.0101   0.0009   9.33e-05  [29s]
  46    0.1061   0.0258    0.0088   0.0008   8.26e-05  [28s]
Best model saved  (Val CER=0.0008)
  47    0.1002   0.0243    0.0104   0.0011   7.24e-05  [28s]
  48    0.0999   0.0240    0.0081   0.0007   6.26e-05  [29s]
Best model saved  (Val CER=0.0007)
  49    0.0963   0.0229    0.0082   0.0010   5.34e-05  [29s]
  50    0.0949   0.0226    0.0079   0.0008   4.48e-05  [28s]
  51    0.0892   0.0217    0.0076   0.0007   3.68e-05  [28s]
Best model saved  (Val CER=0.0007)
  52    0.0895   0.0217    0.0080   0.0007   2.96e-05  [28s]
  53    0.0887   0.0214    0.0067   0.0007   2.30e-05  [31s]
  54    0.0851   0.0209    0.0066   0.0008   1.73e-05  [29s]
  55    0.0878   0.0212    0.0064   0.0006   1.24e-05  [28s]
Best model saved  (Val CER=0.0006)
  56    0.0879   0.0216    0.0062   0.0007   8.32e-06  [28s]
  57  

In [26]:
sub = pd.read_csv(out_csv)
print(f"Submission rows : {len(sub)}")
print(f"Sample predictions:")
print(sub.head(10).to_string(index=False))
print(f"\nPrediction length distribution:")
print(sub["prediction"].str.len().value_counts().sort_index())

Submission rows : 5000
Sample predictions:
        image prediction
   test-0.png     QVTQ8A
   test-1.png     7PSW9D
  test-10.png     7DUP98
 test-100.png     75Z4WT
test-1000.png     QAKZ7V
test-1001.png     R6MERY
test-1002.png     CHXX67
test-1003.png     9NV2WP
test-1004.png     F56TDZ
test-1005.png     FFTFRX

Prediction length distribution:
prediction
4       1
5      22
6    4975
7       2
Name: count, dtype: int64
